# Curse of dimensionality: seeing it with code

Three experiments: more columns can lower accuracy, data gets sparse, and distances stop meaning much.

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.datasets import load_digits
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

## 1. The digits data

1,797 images of handwritten digits, each 8 × 8 pixels, so 64 columns.

In [ ]:
X, y = load_digits(return_X_y=True)
print(X.shape)                       # (1797, 64)
# pixels that never change carry no information
print("constant pixels:", (X.std(axis=0) == 0).sum())

## 2. Accuracy as we add real pixels

We rank pixels by how much they tell us about the digit, then give KNN the best 1, 2, 4, ... of them.

In [ ]:
# most useful pixel first
order = np.argsort(-mutual_info_classif(X, y, random_state=0))

def score(Z):
    # average accuracy over 5 train/test splits
    return cross_val_score(KNeighborsClassifier(), Z, y, cv=5).mean()

for k in [1, 4, 16, 32, 48, 64]:
    print(f"{k:2d} pixels: {score(X[:, order[:k]]):.1%}")

In [ ]:
# leakage check: rank the pixels inside each training part instead of on all images
from functools import partial
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import make_pipeline
mi = partial(mutual_info_classif, random_state=0)
for k in [1, 16, 40]:
    pipe = make_pipeline(SelectKBest(mi, k=k), KNeighborsClassifier())
    print(f"{k:2d} pixels, ranked on training parts only: {cross_val_score(pipe, X, y, cv=5).mean():.1%}")


## 3. Accuracy as we add useless columns

Random numbers on the same 0 to 16 scale as the pixels. They carry no information about the digit.

In [ ]:
rng = np.random.default_rng(0)
noise = rng.uniform(0, 16, (len(X), 400))

rows = []
for n in [0, 50, 100, 200, 400]:
    Z = np.hstack([X, noise[:, :n]])     # 64 pixels + n useless columns
    start = time.perf_counter()
    acc = score(Z)
    rows.append({"columns": 64 + n, "accuracy": acc, "seconds": time.perf_counter() - start})
pd.DataFrame(rows).round(3)

In [ ]:
px.line(pd.DataFrame(rows), x="columns", y="accuracy", markers=True,
        title="More useless columns, lower accuracy")

In [ ]:
# test of Section 4.3: as useless columns are added, is the nearest neighbour still a similar image?
# for each image, find its nearest other image and check whether it shows the same digit
from sklearn.neighbors import NearestNeighbors
for n in [0, 50, 100, 200, 400]:
    Z = np.hstack([X, noise[:, :n]])
    dist, idx = NearestNeighbors(n_neighbors=2).fit(Z).kneighbors(Z)
    same = (y[idx[:, 1]] == y).mean()            # idx[:, 0] is the image itself
    print(f"{64 + n:3d} columns: nearest neighbour shows the same digit for {same:.1%}; "
          f"nearest / mean distance = {(dist[:, 1] / np.linalg.norm(Z[:, None] - Z[None, :200], axis=2).mean(1)).mean():.2f}")

## 4. Sparsity: the same 20 points in more columns

Split each column into 5 bins and count the empty cells.

In [ ]:
pts = np.random.default_rng(0).uniform(0, 1, (20, 3))
bins = np.floor(pts * 5).astype(int)
for d in (1, 2, 3):
    cells = 5 ** d
    used = len({tuple(b[:d]) for b in bins})
    print(f"{d} column(s): {cells:3d} cells, {cells - used:3d} empty ({(cells - used) / cells:.0%})")

## 5. Distances become alike

Distance from one point to 500 others. In many columns, the nearest and farthest are almost the same distance away.

In [ ]:
rng = np.random.default_rng(0)
for d in (2, 10, 100, 1000):
    p = rng.uniform(0, 1, (501, d))
    dist = np.linalg.norm(p[1:] - p[0], axis=1)
    print(f"{d:4d} columns: farthest / nearest = {dist.max() / dist.min():.1f}")